# View 1: nonprofit density vs. need, by nonprofit type (Massachusetts)

**Milestone 4 · Tasks 3d and 4a.** Updates the Milestone 3 density maps with the client's decisions from the
2026-11-13 meeting:

| Decision | What this notebook does |
|---|---|
| Desert cut-off: **quarters** | Desert = top quarter for poverty **and** bottom quarter for nonprofits per 1,000; over-concentration = the reverse |
| Need measure: **poverty rate** | `NEED_VAR = "poverty_rate_pct"` (median income is shown in the hover box) |
| College areas: **flag, don't exclude** | Areas where college and graduate students make up at least 30% of the people in poverty (ACS table B14006) are flagged and outlined, with the poverty rate excluding students shown alongside |
| Break density down **by nonprofit type** | Nonprofits are grouped by NTEE category; the interactive maps switch between types |
| Show results **by town** as well as tract | Organizations are placed in towns directly, and town population and poverty come straight from the ACS |

**Tract vs. town.** Split by type, many tracts (about 4,300 residents each) have no organization of that type,
so a "desert" label would be unstable. So:

| Map | All nonprofits / people-serving | Each nonprofit type |
|---|---|---|
| **Tract** | desert / over-concentration classification | nonprofits per 1,000 residents only (no desert label) |
| **Town** | desert / over-concentration classification | desert / over-concentration classification |

The NTEE grouping is the same one shared with View 3 (`ntee_groups.py` / `ntee_category_map.csv`).

Run top to bottom. Edit only the **Config** cell. Needs a Census API key in Colab Secrets (`CENSUS_API_KEY`).

## Config

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Colab: True = upload the input files from your computer and download the results as a zip.
#        False = read from / save to Google Drive (set DATA_DIR).
UPLOAD_FROM_COMPUTER = True
DATA_DIR = Path("/content/drive/MyDrive") if IN_COLAB else Path("data")
SAVE_DIR_NAME = "view1_density_by_type"

TRACT_FILE = "ma_tract_kpi1.csv"                  # ZIP-fallback / ACS step (task2_outputs)
PLACEMENT_FILE = "ma_org_tract_placement.csv"     # same step: one row per organization with its location
# NTEE codes: any of these (a CSV with EIN and NTEE_CD columns, or bmf_clean.parquet from EIN matching)
NTEE_FILES = ["ma_bmf_ntee.csv", "bmf_clean.parquet"]

NEED_VAR = "poverty_rate_pct"      # client's choice
SPLIT = 1 / 4                      # client's choice: top / bottom quarter
LOW_POP = 500                      # tracts and towns under this many residents are left out of the classification
SMALL_TOWN = 5000                  # towns under this get a note: zero organizations of a type is common there
# College areas (ACS table B14006): flagged when college and graduate students are at least this share of the
# people below the poverty line
STUDENT_POVERTY_SHARE = 0.30
ACS_YEAR = 2024                    # same ACS 5-year release as the tract table (2020-2024)
STATE_FIPS = "25"
MA_ZIP3 = {f"{i:03d}" for i in range(10, 28)} | {"055"}

# NTEE major-group letter -> category. Keep in sync with ntee_groups.py (shared with View 3).
CATEGORIES = [
    ("Human services", "IJKLMOP", "Crime & legal (I), employment (J), food & nutrition (K), housing (L), "
                                   "public safety & disaster relief (M), youth development (O), human services (P)"),
    ("Health", "EFGH", "Health care (E), mental health (F), diseases & disorders (G), medical research (H)"),
    ("Education", "B", "Schools, colleges, libraries, PTAs and other education (B)"),
    ("Arts & culture", "A", "Arts, culture and humanities (A)"),
    ("Environment & animals", "CD", "Environment (C), animal-related (D)"),
    ("Recreation & sports", "N", "Recreation, sports and leisure (N)"),
    ("Religion", "X", "Religion-related (X); most churches don't file a 990, so this is undercounted"),
    ("Community & philanthropy", "STW", "Community improvement (S), philanthropy & grantmaking (T), "
                                         "public & societal benefit (W)"),
    ("Other", "QRUVYZ", "International (Q), civil rights (R), science & technology (U), social science (V), "
                          "mutual & membership benefit (Y), unknown (Z)"),
]
UNKNOWN = "Unknown (no NTEE code)"
PRESETS = {"People-serving (human services + health)": ["Human services", "Health"]}
MAP_CATEGORIES = [c for c, _, _ in CATEGORIES if c != "Other"]   # Other and Unknown are counted, not mapped

## Setup

In [ ]:
import subprocess

if IN_COLAB:
    if UPLOAD_FROM_COMPUTER:
        from google.colab import files
        UP = Path("/content/upload")
        UP.mkdir(exist_ok=True)
        missing = [f for f in [TRACT_FILE, PLACEMENT_FILE] if not (UP / f).exists()]
        if not any((UP / f).exists() for f in NTEE_FILES):
            missing.append(" or ".join(NTEE_FILES))
        if missing:
            print("Choose these files from your computer (you can select several at once):\n  " + ", ".join(missing))
            for fname, data in files.upload().items():
                (UP / fname).write_bytes(data)
        DATA_DIR = UP
    else:
        from google.colab import drive
        drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas", "folium", "pyarrow"], check=True)

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)


def locate(name: str, required: bool = True):
    p = DATA_DIR / name
    if p.exists():
        return p
    if DATA_DIR.exists():
        hit = next(DATA_DIR.rglob(name), None)
        if hit:
            return hit
    if required:
        raise FileNotFoundError(f"Could not find {name} under {DATA_DIR}. Set DATA_DIR in the Config cell.")
    return None


TRACT_PATH = locate(TRACT_FILE)
PLACEMENT_PATH = locate(PLACEMENT_FILE)
NTEE_PATH = next((p for p in (locate(f, required=False) for f in NTEE_FILES) if p), None)
if NTEE_PATH is None:
    raise FileNotFoundError(f"Could not find any of {NTEE_FILES} under {DATA_DIR}.")
SAVE_DIR = (Path("/content") if IN_COLAB and UPLOAD_FROM_COMPUTER else TRACT_PATH.parent.parent) / SAVE_DIR_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
for k, v in {"tracts": TRACT_PATH, "placement": PLACEMENT_PATH, "ntee": NTEE_PATH}.items():
    print(f"{k:10s} {v}")
print("Outputs go to", SAVE_DIR)

LETTER_TO_CAT = {ch: name for name, letters, _ in CATEGORIES for ch in letters}
CAT_DESC = {name: desc for name, _, desc in CATEGORIES}
ALL = "All nonprofits"
FILTERS = [ALL] + list(PRESETS) + MAP_CATEGORIES
TRACT_CLASSIFIED = [ALL] + list(PRESETS)          # tract maps label deserts only for these


def categorize(ntee_cd: pd.Series) -> pd.Series:
    letter = ntee_cd.fillna("").astype(str).str.strip().str.upper().str[:1]
    return letter.map(LETTER_TO_CAT).fillna(UNKNOWN)


def slug(name: str) -> str:
    s = "".join(ch if ch.isalnum() else "_" for ch in name.lower())
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")

## 1. Load tracts, organizations and NTEE codes

Organizations are the ones placed by the ZIP-fallback / ACS step (geocoded address, or ZIP fallback for PO boxes
and failed addresses). Each gets an NTEE category from the IRS Business Master File. About a quarter have no NTEE
code: they count toward **All nonprofits** but not toward any single type.

In [ ]:
t = pd.read_csv(TRACT_PATH, dtype={"geoid": str, "county_fips": str})
t["geoid"] = t.geoid.str.zfill(11)
for c in ["population", "median_household_income", "poverty_rate_pct", "median_age", "nonprofit_count"]:
    t[c] = pd.to_numeric(t[c], errors="coerce")

org = pd.read_csv(PLACEMENT_PATH, dtype=str)
org = org[org.placement_method != "unplaced"].copy()
org["final_tract_geoid"] = org.final_tract_geoid.str.zfill(11)
org["zip5"] = org.zip5.fillna("").str[:5].str.zfill(5)
org[["latitude", "longitude"]] = org[["latitude", "longitude"]].apply(pd.to_numeric, errors="coerce")

if NTEE_PATH.suffix == ".parquet":
    bmf = pd.read_parquet(NTEE_PATH, columns=["EIN", "NTEE_CD"])
else:
    bmf = pd.read_csv(NTEE_PATH, dtype=str, keep_default_na=False, usecols=["EIN", "NTEE_CD"])
bmf["EIN"] = bmf.EIN.astype(str).str.zfill(9)
org = org.merge(bmf.drop_duplicates("EIN"), left_on="ein", right_on="EIN", how="left")
org["category"] = categorize(org.NTEE_CD)

cat_counts = org.category.value_counts().rename("organizations").to_frame()
cat_counts["share_%"] = (100 * cat_counts.organizations / len(org)).round(1)
print(f"Placed organizations: {len(org):,} (tract table total: {int(t.nonprofit_count.sum()):,})")
print(org.placement_method.value_counts().to_string())
print(cat_counts.to_string())


def count_by(org_df: pd.DataFrame, key: str, index) -> pd.DataFrame:
    """Organizations per area for every filter (all, presets, each category, unknown)."""
    wide = org_df.pivot_table(index=key, columns="category", values="ein", aggfunc="count", fill_value=0)
    out = pd.DataFrame(index=pd.Index(index, name=key))
    out[ALL] = org_df.groupby(key).size()
    for name, members in PRESETS.items():
        out[name] = wide.reindex(columns=members, fill_value=0).sum(axis=1)
    for c in [c for c, _, _ in CATEGORIES] + [UNKNOWN]:
        out[c] = wide[c] if c in wide else 0
    return out.fillna(0).astype(int)


def add_rates(df: pd.DataFrame, counts: pd.DataFrame, key: str) -> pd.DataFrame:
    for f in counts.columns:
        df[f"n_{slug(f)}"] = df[key].map(counts[f]).fillna(0).astype(int).values
        df[f"per1k_{slug(f)}"] = np.where(df.population > 0, 1000 * df[f"n_{slug(f)}"] / df.population, np.nan)
    return df

## 2. Census helpers

Census table columns have codes like `B14006_009E` that don't say what they hold, and a mistyped code would
silently pull the wrong numbers. So each column is picked by its **Census label** (for example "Income below
poverty level → Enrolled in college undergraduate years"); if exactly one column doesn't match, the code stops.
The chosen columns and labels are printed so they can be checked by eye.

In [ ]:
import os
import requests

BASE = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"


def census_key() -> str:
    key = os.environ.get("CENSUS_API_KEY", "")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("CENSUS_API_KEY") or ""
        except Exception:
            pass
    return key.strip().strip('"').strip("'").strip()


KEY = census_key()
print("Census API key length:", len(KEY), "(should be 40)")
_label_cache = {}


def labels(group: str) -> dict:
    if group not in _label_cache:
        r = requests.get(f"{BASE}/groups/{group}.json", timeout=120)
        r.raise_for_status()
        _label_cache[group] = {v: d["label"] for v, d in r.json()["variables"].items() if v.endswith("E")}
    return _label_cache[group]


def pick(group: str, *parts: str) -> str:
    """The one estimate in `group` whose label path is exactly `parts` (after 'Estimate')."""
    # Census wording varies between tables ("below poverty level" vs. "below the poverty level"), so ignore "the"
    norm = lambda s: " ".join(w for w in s.strip(" :").lower().split() if w != "the")
    hits = [v for v, lab in labels(group).items()
            if [norm(p) for p in lab.split("!!")[1:]] == [norm(p) for p in parts]]
    if len(hits) != 1:
        raise ValueError(f"Expected one {group} variable for {parts}, found {hits}")
    return hits[0]


def pick_prefix(group: str, prefix: str) -> str:
    hits = [v for v, lab in labels(group).items() if lab.split("!!")[-1].lower().startswith(prefix.lower())]
    if len(hits) != 1:
        raise ValueError(f"Expected one {group} variable starting with {prefix!r}, found {hits}")
    return hits[0]


def acs(variables: dict, geo_for: str) -> pd.DataFrame:
    """Pull named ACS variables for every `geo_for` area in Massachusetts. Negative codes become missing."""
    params = [("get", ",".join(variables.values())), ("for", f"{geo_for}:*"), ("in", f"state:{STATE_FIPS}"),
              ("in", "county:*")] + ([("key", KEY)] if KEY else [])
    r = requests.get(BASE, params=params, timeout=300)
    if not r.ok or not r.text.lstrip().startswith("["):
        raise RuntimeError(f"ACS request failed (HTTP {r.status_code}): {r.text[:200]}. "
                           "Check CENSUS_API_KEY in Colab Secrets.")
    rows = r.json()
    df = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns={v: k for k, v in variables.items()})
    df[list(variables)] = df[list(variables)].apply(pd.to_numeric, errors="coerce")
    df[list(variables)] = df[list(variables)].where(df[list(variables)] >= 0)
    return df


BELOW = "Income in the past 12 months below poverty level"
ABOVE = "Income in the past 12 months at or above poverty level"
V_STUDENTS = {
    "b14_total": pick("B14006", "Total"),
    "b14_below": pick("B14006", "Total", BELOW),
    "below_undergrad": pick("B14006", "Total", BELOW, "Enrolled in school", "Enrolled in college undergraduate years"),
    "below_grad": pick("B14006", "Total", BELOW, "Enrolled in school", "Enrolled in graduate or professional school"),
    "above_undergrad": pick("B14006", "Total", ABOVE, "Enrolled in school", "Enrolled in college undergraduate years"),
    "above_grad": pick("B14006", "Total", ABOVE, "Enrolled in school", "Enrolled in graduate or professional school"),
}
V_TOWN = {
    "population": pick("B01003", "Total"),
    "poverty_universe": pick("B17001", "Total"),
    "poverty_below": pick("B17001", "Total", BELOW),
    "median_household_income": pick_prefix("B19013", "Median household income"),
}
for k, v in {**V_STUDENTS, **V_TOWN}.items():
    print(f"{k:24s} {v:14s} {labels(v.split('_')[0])[v]}")


def add_student_measures(df: pd.DataFrame) -> pd.DataFrame:
    below_students = df.below_undergrad + df.below_grad
    all_students = below_students + df.above_undergrad + df.above_grad
    df["college_students"] = all_students
    df["student_share_of_poverty_%"] = (100 * below_students / df.b14_below.replace(0, np.nan)).round(1)
    df["poverty_rate_excl_students_pct"] = (100 * (df.b14_below - below_students)
                                            / (df.b14_total - all_students).replace(0, np.nan)).round(2)
    df["college_student_share_%"] = (100 * all_students / df.b14_total.replace(0, np.nan)).round(1)
    return df


STUDENT_COLS = ["college_students", "student_share_of_poverty_%", "poverty_rate_excl_students_pct",
                "college_student_share_%"]

## 3. Tracts: density by type, college flag, classification

**College areas.** Off-campus college students usually report little income, so they raise the poverty rate
(students in dorms are not counted in poverty statistics). A tract is flagged when college and graduate students
are at least `STUDENT_POVERTY_SHARE` of its people in poverty. Flagged tracts **stay in** the classification, as
the client asked; the flag and the poverty rate excluding students explain why their rate is high.

**Classification** (tracts with 500+ residents), only for **All nonprofits** and the **people-serving** preset:

| Group | Rule |
|---|---|
| Possible nonprofit desert | poverty in the top quarter, density in the bottom quarter |
| Possible over-concentration | poverty in the bottom quarter, density in the top quarter |
| High need, many nonprofits | poverty in the top quarter, density in the top quarter |
| Other | everything else |

**Gap score** (same areas and filters): poverty rank minus density rank, each a percentile from 0 to 100.
Positive = more need than nonprofits (toward a desert); negative = more nonprofits than need (toward
over-concentration); near 0 = need and nonprofits are similar. It has no cut-off, so areas just outside the
top or bottom quarter still show as close to a desert or to over-concentration.

In [ ]:
t = add_rates(t, count_by(org, "final_tract_geoid", t.geoid), "geoid")
t["low_population_flag"] = t.population < LOW_POP

b14 = acs(V_STUDENTS, "tract")
b14["geoid"] = b14.state + b14.county + b14.tract
b14 = add_student_measures(b14)
t = t.drop(columns=[c for c in STUDENT_COLS if c in t]).merge(b14[["geoid"] + STUDENT_COLS], on="geoid", how="left")
t["college_area_flag"] = (t["student_share_of_poverty_%"] >= 100 * STUDENT_POVERTY_SHARE) & ~t.low_population_flag
print(f"Tracts matched to B14006: {t['student_share_of_poverty_%'].notna().sum():,} of {len(t):,}")
print(f"College-area tracts: {t.college_area_flag.sum()}")
print(t[t.college_area_flag].sort_values("student_share_of_poverty_%", ascending=False)
      [["tract_name", "population", "poverty_rate_pct", "poverty_rate_excl_students_pct",
        "student_share_of_poverty_%"]].head(12).to_string(index=False))

GROUPS = ["Possible nonprofit desert", "Possible over-concentration", "High need, many nonprofits", "Other"]
LOWPOP = "Low population (excluded)"
COLORS = {GROUPS[0]: "#D9822B", GROUPS[1]: "#2F6690", GROUPS[2]: "#4CA394", GROUPS[3]: "#C9C9C9", LOWPOP: "#EDEDED"}


def classify(df: pd.DataFrame, filt: str, unit: str) -> dict:
    """Add group_<filter> (areas with population >= LOW_POP are classified) and return a summary row."""
    ok = df.population >= LOW_POP
    s = df[ok]
    need, dens = s[NEED_VAR], s[f"per1k_{slug(filt)}"]
    q_need_hi, q_need_lo = need.quantile(1 - SPLIT), need.quantile(SPLIT)
    q_dens_hi, q_dens_lo = dens.quantile(1 - SPLIT), dens.quantile(SPLIT)
    hi_need, lo_need = need >= q_need_hi, need <= q_need_lo
    hi_dens, lo_dens = (dens >= q_dens_hi) & (dens > 0), dens <= q_dens_lo
    g = pd.Series(np.select([hi_need & lo_dens, lo_need & hi_dens, hi_need & hi_dens], GROUPS[:3], GROUPS[3]),
                  index=s.index)
    col = f"group_{slug(filt)}"
    df[col] = LOWPOP
    df.loc[ok, col] = g
    df[f"gap_{slug(filt)}"] = np.nan
    df.loc[ok, f"gap_{slug(filt)}"] = (100 * (need.rank(pct=True) - dens.rank(pct=True))).round(1)
    d, o = df[col] == GROUPS[0], df[col] == GROUPS[1]
    n = df[f"n_{slug(filt)}"]
    return {"filter": filt, f"desert_{unit}": int(d.sum()), "desert_residents": int(df.population[d].sum()),
            "desert_per_1k": round(1000 * n[d].sum() / max(df.population[d].sum(), 1), 2),
            f"overconc_{unit}": int(o.sum()), "overconc_residents": int(df.population[o].sum()),
            "overconc_per_1k": round(1000 * n[o].sum() / max(df.population[o].sum(), 1), 2),
            "poverty_cutoff_top_%": round(q_need_hi, 1), "density_cutoff_bottom": round(q_dens_lo, 2),
            "density_cutoff_top": round(q_dens_hi, 2), f"{unit}_with_zero_%": round(100 * (dens == 0).mean(), 1),
            f"desert_{unit}_college_flagged": int((df.get("college_area_flag", False) & d).sum())}


tract_summary = pd.DataFrame([classify(t, f, "tracts") for f in TRACT_CLASSIFIED]).set_index("filter")
print(tract_summary.to_string())

stable = t[~t.low_population_flag]
type_density = pd.DataFrame({f: {"statewide_per_1k": round(1000 * t[f"n_{slug(f)}"].sum() / t.population.sum(), 2),
                                 "median_tract_per_1k": round(stable[f"per1k_{slug(f)}"].median(), 2),
                                 "tracts_with_zero_%": round(100 * (stable[f"n_{slug(f)}"] == 0).mean(), 1)}
                             for f in FILTERS}).T
print("\nDensity by type (tracts with 500+ residents):")
print(type_density.to_string())

## 4. Boundaries

Census cartographic boundary files: MA tracts (2020 geography, used by the 2020–2024 ACS), MA county
subdivisions (the 351 cities and towns) and ZIP Code Tabulation Areas (ZCTAs).

In [ ]:
import io, zipfile
import geopandas as gpd

CACHE = Path("/content/geo_cache") if IN_COLAB else Path("geo_cache")
CACHE.mkdir(parents=True, exist_ok=True)


def fetch(urls):
    for url in urls:
        folder = CACHE / Path(url).stem
        try:
            if not folder.exists():
                print("  downloading", Path(url).name, "...")
                r = requests.get(url, timeout=600)
                r.raise_for_status()
                zipfile.ZipFile(io.BytesIO(r.content)).extractall(folder)
            print("Boundaries:", url)
            return gpd.read_file(next(folder.glob("*.shp")))
        except Exception as e:
            print("Failed:", url, e)
    raise RuntimeError("Could not download boundaries")


YEARS = (2024, 2023, 2022, 2020)
tracts = fetch([f"https://www2.census.gov/geo/tiger/GENZ{y}/shp/cb_{y}_{STATE_FIPS}_tract_500k.zip"
                for y in YEARS])[["GEOID", "geometry"]]
towns = fetch([f"https://www2.census.gov/geo/tiger/GENZ{y}/shp/cb_{y}_{STATE_FIPS}_cousub_500k.zip" for y in YEARS])
towns = towns[towns.ALAND.astype(float) > 0][["GEOID", "NAME", "geometry"]].rename(
    columns={"GEOID": "town_geoid", "NAME": "town"})
zcta = fetch(["https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_zcta520_500k.zip"])
zcol = [c for c in zcta.columns if c.upper().startswith("ZCTA5")][0]
zcta = zcta[zcta[zcol].str[:3].isin(MA_ZIP3)][[zcol, "geometry"]].rename(columns={zcol: "zip5"})
print(f"Tracts: {len(tracts):,} | towns: {len(towns):,} | MA ZCTAs: {len(zcta):,}")

## 5. Place organizations in towns

- **Geocoded organizations:** the town whose boundary contains the address point (nearest town if a point falls
  just outside, for example on the shoreline).
- **ZIP-fallback organizations** (PO boxes, failed addresses): the town that covers the largest part of their ZIP
  code area. Massachusetts ZIP codes mostly follow town lines, while rural tracts can span several small towns.
  ZIPs with no ZCTA (PO-box-only ZIPs) use the center of their assigned tract.

In [ ]:
SP = 26986   # Massachusetts State Plane, meters
towns_p = towns.to_crs(SP)

geo = org[(org.placement_method == "geocoded") & org.latitude.notna()]
pts = gpd.GeoDataFrame(geo[["ein"]], geometry=gpd.points_from_xy(geo.longitude, geo.latitude), crs=4326).to_crs(SP)
inside = gpd.sjoin(pts, towns_p, how="left", predicate="within")
inside = inside[~inside.index.duplicated()]
miss = inside.town_geoid.isna()
if miss.any():
    near = gpd.sjoin_nearest(pts[miss], towns_p, how="left")
    near = near[~near.index.duplicated()].reindex(inside.index[miss])
    inside.loc[miss, ["town_geoid", "town"]] = near[["town_geoid", "town"]].values
geo_town = inside[["ein", "town_geoid", "town"]].assign(town_method=np.where(miss, "nearest_town", "point_in_town"))

zp = zcta.to_crs(SP)
zp["zcta_area"] = zp.area
zov = gpd.overlay(zp, towns_p, how="intersection", keep_geom_type=True)
zov["share"] = zov.area / zov.zcta_area
zip_town = zov.sort_values("share", ascending=False).drop_duplicates("zip5")[["zip5", "town_geoid", "town", "share"]]

fb = org[org.placement_method != "geocoded"][["ein", "zip5", "final_tract_geoid"]]
fb = fb.merge(zip_town, on="zip5", how="left")
no_zcta = fb.town_geoid.isna()
if no_zcta.any():
    cent = tracts.to_crs(SP).set_index("GEOID").centroid
    cpts = gpd.GeoDataFrame(fb.loc[no_zcta, ["ein"]], geometry=fb.loc[no_zcta, "final_tract_geoid"].map(cent),
                            crs=SP)
    cpts = cpts[cpts.geometry.notna()]
    cj = gpd.sjoin_nearest(cpts, towns_p, how="left")
    cj = cj[~cj.index.duplicated()]
    fb.loc[cj.index, ["town_geoid", "town"]] = cj[["town_geoid", "town"]].values
fb["town_method"] = np.where(no_zcta, "tract_center", "zip_largest_overlap")
org_town = pd.concat([geo_town, fb[["ein", "town_geoid", "town", "town_method"]]], ignore_index=True)
org = org.drop(columns=[c for c in ["town_geoid", "town", "town_method"] if c in org]).merge(org_town, on="ein", how="left")
print(org.town_method.value_counts(dropna=False).to_string())
print(f"Organizations placed in a town: {org.town_geoid.notna().sum():,} of {len(org):,}")

## 6. Towns: ACS values, density by type, classification

Town population, poverty rate, median income and student measures come **directly from the ACS** for county
subdivisions (the same 2020–2024 release), not from adding up tracts. Every nonprofit type is classified at the
town level, among towns with 500+ residents.

In [ ]:
tacs = acs({**V_TOWN, **V_STUDENTS}, "county subdivision")
tacs["town_geoid"] = tacs.state + tacs.county + tacs["county subdivision"]
tacs["poverty_rate_pct"] = (100 * tacs.poverty_below / tacs.poverty_universe.replace(0, np.nan)).round(2)
tacs = add_student_measures(tacs)
town = towns[["town_geoid", "town"]].merge(
    tacs[["town_geoid", "population", "poverty_rate_pct", "median_household_income"] + STUDENT_COLS],
    on="town_geoid", how="left")
town = add_rates(town, count_by(org.dropna(subset=["town_geoid"]), "town_geoid", town.town_geoid), "town_geoid")
town["low_population_flag"] = town.population.fillna(0) < LOW_POP
town["college_area_flag"] = (town["student_share_of_poverty_%"] >= 100 * STUDENT_POVERTY_SHARE) & ~town.low_population_flag
town["population"] = town.population.fillna(0)
town_summary = pd.DataFrame([classify(town, f, "towns") for f in FILTERS]).set_index("filter")
print(f"Towns: {len(town)} | under {LOW_POP} residents: {town.low_population_flag.sum()} | "
      f"college-area towns: {town.college_area_flag.sum()} | ACS population total: {town.population.sum():,.0f}")
print(town_summary.to_string())
print("\nCollege-area towns:", ", ".join(town[town.college_area_flag].sort_values(
    "student_share_of_poverty_%", ascending=False).town.head(15)))

## 7. Static map: all nonprofits vs. people-serving nonprofits (tracts)

Side by side, to show how deserts change once land conservancies, animal welfare groups and other
non-people-serving organizations are taken out. College-area tracts are outlined in black.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

gt = tracts.merge(t, left_on="GEOID", right_on="geoid", how="inner")
college_t = gt[gt.college_area_flag.fillna(False).astype(bool)]
college_label = f"College area (students >= {STUDENT_POVERTY_SHARE:.0%} of people in poverty)"

preset = list(PRESETS)[0]
fig, axes = plt.subplots(1, 2, figsize=(22, 8))
for ax, f in zip(axes, [ALL, preset]):
    col, counts_ = f"group_{slug(f)}", t[f"group_{slug(f)}"].value_counts()
    for name, color in COLORS.items():
        d = gt[gt[col] == name]
        if len(d):
            d.plot(ax=ax, color=color, edgecolor="#FFFFFF", linewidth=0.1)
    if len(college_t):
        college_t.boundary.plot(ax=ax, color="#111111", linewidth=0.6)
    ax.set_axis_off()
    ax.set_title(f, fontsize=14)
    ax.legend(handles=[Patch(color=c, label=f"{n} ({counts_.get(n, 0):,})") for n, c in COLORS.items()]
              + [Line2D([], [], color="#111111", lw=1, label=college_label)], loc="lower left", frameon=False,
              fontsize=9)
fig.suptitle("Possible nonprofit deserts vs. over-concentration by census tract, Massachusetts "
             "(quarters of poverty rate and nonprofits per 1,000)", fontsize=16)
fig.text(0.05, 0.02, "Nonprofits are placed at their filing address. About a quarter have no NTEE code and appear "
         "only under All nonprofits.", fontsize=10, color="#555555")
fig.savefig(SAVE_DIR / "map_tract_deserts_all_vs_people_serving.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Interactive maps

Pick a layer in the menu (top right); hover an area for its numbers. The panel at the bottom left explains the
colors and what each nonprofit type includes.

- **Tract map:** All nonprofits and People-serving show desert / over-concentration groups. Each nonprofit type
  shows nonprofits of that type per 1,000 residents (darker = more; cream = none of that type).
- **Town map:** every layer shows desert / over-concentration groups.
- **Gap score layers** (wherever there are desert groups): poverty rank minus density rank, shaded from dark blue
  (more nonprofits than need) through near-white (balanced) to dark orange (more need than nonprofits).

Colors mean the same thing on every layer: orange = gap, blue = more nonprofits, teal = high need and many
nonprofits, gray = other or excluded.

In [ ]:
import json
import folium

TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ATTR = "Tiles &copy; Esri &mdash; Esri, HERE, Garmin, OpenStreetMap contributors"
DENS_COLORS = ["#F6F1E1", "#D6E2EE", "#A3BED8", "#6D94BC", "#2F6690"]   # none, then four blues
GAP_EDGES = [-60, -30, -10, 10, 30, 60]                                   # gap score bins (-100 to 100)
GAP_COLORS = ["#2F6690", "#7FA3C4", "#C8D7E6", "#F4F1EA", "#F2CFA8", "#E8A564", "#D9822B"]
GAP_LABELS = ["-100 to -60", "-60 to -30", "-30 to -10", "-10 to 10", "10 to 30", "30 to 60", "60 to 100"]


def gap_colors(values: pd.Series) -> np.ndarray:
    """Blue (more nonprofits than need) -> near-white (balanced) -> orange (more need); gray if not scored."""
    col = np.array(GAP_COLORS)[np.searchsorted(GAP_EDGES, values.fillna(0), side="right")]
    return np.where(values.isna(), COLORS[LOWPOP], col)


def density_colors(values: pd.Series, lowpop: pd.Series):
    """Cream for zero; quartiles of the positive values in four blues; low-population areas gray."""
    pos = values[(values > 0) & ~lowpop]
    edges = np.unique(np.quantile(pos, [0, .25, .5, .75, 1])) if len(pos) else np.array([0, 1])
    steps = np.clip(np.searchsorted(edges, values, side="right") - 1, 0, len(edges) - 2) + 1
    col = np.where(values > 0, np.array(DENS_COLORS)[np.minimum(steps, 4)], DENS_COLORS[0])
    col = np.where(lowpop, COLORS[LOWPOP], col)
    labels_ = ["None of this type"] + [f"{edges[i]:.2f}–{edges[i + 1]:.2f}" for i in range(len(edges) - 1)]
    return col, labels_


def legend_html(unit: str, has_density: bool) -> str:
    swatch = lambda c: f"<span style='display:inline-block;width:12px;height:12px;background:{c};margin-right:6px'></span>"
    groups = "".join(f"<div>{swatch(c)}{n}</div>" for n, c in COLORS.items())
    dens = ("<div style='margin-top:6px'><b>Single-type layers</b>: nonprofits of that type per 1,000 residents"
            "<div>" + "".join(swatch(c) for c in DENS_COLORS) + " none → more</div></div>") if has_density else ""
    which = ("All nonprofits and People-serving layers" if has_density else "Every layer")
    gap = ("<div style='margin-top:6px'><b>Gap score layers</b>: poverty rank minus nonprofit density rank "
           "(each 0&ndash;100)" + "".join(f"<div>{swatch(c)}{l}</div>" for c, l in zip(GAP_COLORS, GAP_LABELS))
           + "<div style='color:#555'>Blue = more nonprofits than need; orange = more need than nonprofits</div></div>")
    types = "".join(f"<div style='margin-top:4px'><b>{k}</b>: {v}</div>"
                    for k, v in [(ALL, "every placed organization, including those with no NTEE code")]
                    + [(p, " + ".join(m)) for p, m in PRESETS.items()] + [(c, CAT_DESC[c]) for c in MAP_CATEGORIES])
    return ("<div style='position:fixed;bottom:16px;left:16px;z-index:9999;background:white;padding:10px 12px;"
            "border:1px solid #bbb;border-radius:6px;font:12px sans-serif;max-width:340px;max-height:70vh;overflow:auto'>"
            f"<b>Nonprofit density vs. need ({unit})</b><div style='margin-top:4px'><b>{which}</b>: quarters of "
            f"poverty rate and nonprofits per 1,000</div>{groups}{gap}{dens}"
            "<div style='margin-top:4px'><span style='display:inline-block;width:12px;height:0;border-top:2px solid #111;"
            f"margin-right:6px'></span>College area (students &ge; {STUDENT_POVERTY_SHARE:.0%} of people in poverty)</div>"
            f"<details style='margin-top:6px'><summary><b>What each type includes</b></summary>{types}"
            "<div style='margin-top:6px;color:#555'>About a quarter of organizations have no NTEE code and appear only "
            "under All nonprofits. Organizations are placed at their filing address.</div></details>"
            + (f"<div style='margin-top:6px;color:#555'><b>Small towns</b>: towns under {SMALL_TOWN:,} residents often "
               "have no organization of a given type. Organizations in nearby cities may serve these areas.</div>"
               if unit == "town" else "") + "</div>")


def interactive(gdf, name_col, unit, classified, out_name, simplify):
    g = gdf.to_crs(4326).copy()
    g["geometry"] = g.geometry.simplify(simplify, preserve_topology=True)
    g["residents"] = g.population.fillna(0).astype(int)
    g["poverty"] = g.poverty_rate_pct.round(1)
    g["pov_excl_students"] = g.poverty_rate_excl_students_pct.round(1)
    g["income"] = g.median_household_income.map(lambda v: "" if pd.isna(v) else f"${v:,.0f}")
    g["college"] = np.where(g.college_area_flag.fillna(False).astype(bool), "Yes", "")
    g["note"] = np.where(g.residents < SMALL_TOWN, "Small town; nearby organizations may serve this area", "")
    lowpop = g.low_population_flag.fillna(True).astype(bool)
    m = folium.Map(location=[42.25, -71.8], zoom_start=8, tiles=None)
    folium.TileLayer(TILES, attr=ATTR, max_zoom=16, control=False).add_to(m)
    # each classified filter gets a group layer followed by its gap score layer
    layers = [(f, "group" if f in classified else "density") for f in FILTERS]
    layers = [x for f, k in layers for x in ([(f, k), (f, "gap")] if k == "group" else [(f, k)])]
    for i, (f, kind) in enumerate(layers):
        s = slug(f)
        g["_n"] = g[f"n_{s}"].fillna(0).astype(int)
        g["_per1k"] = g[f"per1k_{s}"].round(2)
        g["_gap"] = g[f"gap_{s}"].round(0) if kind != "density" else np.nan
        if kind == "density":
            g["_color"], _ = density_colors(g[f"per1k_{s}"].fillna(0), lowpop)
            g["_group"] = "Density only (no desert label at this level)"
        else:
            g["_group"] = g[f"group_{s}"]
            g["_color"] = (g._group.map(COLORS).fillna(COLORS[LOWPOP]) if kind == "group"
                           else gap_colors(g[f"gap_{s}"]))
        fields = [name_col, "_group", "_n", "_per1k", "residents", "poverty", "pov_excl_students", "income", "college"]
        aliases = [unit.capitalize(), "Group", f, "Per 1,000", "Residents", "Poverty %", "Poverty % excl. students",
                   "Median income", "College area"]
        if kind != "density":
            fields.insert(2, "_gap")
            aliases.insert(2, "Gap score (-100 to 100)")
        if unit == "town":
            fields, aliases = fields + ["note"], aliases + ["Note"]
        name = f if kind != "gap" else f"{f}: gap score"
        layer = folium.FeatureGroup(name=name, overlay=False, show=(i == 0))
        folium.GeoJson(json.loads(g[fields + ["_color", "geometry"]].to_json()),
                       style_function=lambda ft: {"fillColor": ft["properties"]["_color"], "color": "#FFFFFF",
                                                  "weight": 0.3, "fillOpacity": 0.8},
                       tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases)).add_to(layer)
        layer.add_to(m)
    cl = g[g.college == "Yes"][["geometry"]]
    if len(cl):
        folium.GeoJson(json.loads(cl.to_json()), name="College areas (outline)",
                       style_function=lambda ft: {"fill": False, "color": "#111111", "weight": 1.5}).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(folium.Element(legend_html(unit, has_density=len(classified) < len(FILTERS))))
    m.save(str(SAVE_DIR / out_name))
    print(f"Saved {out_name} ({(SAVE_DIR / out_name).stat().st_size / 1e6:.1f} MB)")
    return m


# Simplification tolerance in degrees: 0.0005 is about 50 m (invisible statewide; slightly rougher when zoomed in)
m_tract = interactive(gt, "tract_name", "tract", TRACT_CLASSIFIED, "view1_tracts.html", simplify=0.0005)
gtown = towns.merge(town, on=["town_geoid", "town"], how="inner")
m_town = interactive(gtown, "town", "town", FILTERS, "view1_towns.html", simplify=0.0005)
display(m_town)

## 9. Save

In [ ]:
keep = (["geoid", "tract_name", "county_fips", "population", "poverty_rate_pct", "median_household_income",
         "median_age", "low_population_flag", "college_area_flag"] + STUDENT_COLS
        + [c for c in t.columns if c.startswith(("n_", "per1k_", "group_", "gap_"))])
t[keep].to_csv(SAVE_DIR / "tracts_by_type.csv", index=False)
town.to_csv(SAVE_DIR / "towns_by_type.csv", index=False)
org[["ein", "category", "final_tract_geoid", "town_geoid", "town", "town_method", "placement_method"]].to_csv(
    SAVE_DIR / "org_tract_town.csv", index=False)
tract_summary.to_csv(SAVE_DIR / "desert_summary_tracts.csv")
town_summary.to_csv(SAVE_DIR / "desert_summary_towns.csv")
type_density.to_csv(SAVE_DIR / "density_by_type_tracts.csv")
cat_counts.to_csv(SAVE_DIR / "organizations_by_category.csv")
pd.DataFrame([{"filter": f, "includes": CAT_DESC.get(f) or (" + ".join(PRESETS[f]) if f in PRESETS
                                                            else "Every placed organization"),
               "tract_map": "desert classification + gap score" if f in TRACT_CLASSIFIED else "density only",
               "town_map": "desert classification + gap score"} for f in FILTERS]).to_csv(
    SAVE_DIR / "filter_definitions.csv", index=False)
for p in sorted(SAVE_DIR.iterdir()):
    print(f"{p.name:40s} {p.stat().st_size / 1e3:9.1f} KB")

if IN_COLAB and UPLOAD_FROM_COMPUTER:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive(f"/content/{SAVE_DIR_NAME}", "zip", SAVE_DIR))